# Voicebox Backend on Google Colab T4

Este notebook levanta **todo el backend** en Colab para que Qwen3-TTS genere audio en remoto.

## Flujo
1. Configurar parámetros
2. Instalar dependencias
3. (Opcional) montar Google Drive
4. Levantar backend en segundo plano
5. Exponer URL pública con ngrok
6. Validar con smoke tests


In [ ]:
!nvidia-smi

In [ ]:
# ===== Configuracion =====
REPO_URL = "https://github.com/cdryampi/voicebox.git"
REPO_REF = "feat/colab-remote-api-key"  # branch/tag/commit

# Seguridad: NO pongas claves reales en este archivo.
# Puedes dejar estos valores en None y se pediran en runtime.
VOICEBOX_API_KEY = None
VOICEBOX_GROQ_API_KEY = None  # requerido para STT remoto
VOICEBOX_GROQ_STT_MODEL = "whisper-large-v3-turbo"
NGROK_AUTH_TOKEN = None       # opcional

# Persistencia
USE_GOOGLE_DRIVE = False
DRIVE_DATA_DIR = "/content/drive/MyDrive/voicebox-data"
LOCAL_DATA_DIR = "/content/voicebox-data"

# Backend
VOICEBOX_HOST = "0.0.0.0"
VOICEBOX_PORT = "17493"
VOICEBOX_DEFAULT_MODEL_SIZE = "1.7B"
VOICEBOX_STT_PROVIDER = "groq"
VOICEBOX_STT_REMOTE_NO_FALLBACK = "true"
VOICEBOX_ALLOWED_ORIGINS = "http://localhost:5174,http://127.0.0.1:5174"
VOICEBOX_DB_USE_NULL_POOL = "true"



In [ ]:
# ===== Clone + install =====
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path('/content/voicebox')

if repo_dir.exists() and not (repo_dir / '.git').exists():
    if any(repo_dir.iterdir()):
        print('Found non-git directory at', repo_dir, '- removing it before clone')
        shutil.rmtree(repo_dir)
    else:
        repo_dir.rmdir()

if not (repo_dir / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
else:
    print('Repository already exists at', repo_dir)

os.chdir(repo_dir)

subprocess.run(['git', 'fetch', '--all', '--tags', '--prune'], check=True)

# Ensure local branch/working tree is exactly up to date with origin/<REPO_REF> when available
origin_ref = f'origin/{REPO_REF}'
has_origin_ref = subprocess.run(
    ['git', 'rev-parse', '--verify', '--quiet', origin_ref],
    capture_output=True,
    text=True,
).returncode == 0

if has_origin_ref:
    subprocess.run(['git', 'checkout', '-B', REPO_REF, origin_ref], check=True)
    subprocess.run(['git', 'reset', '--hard', origin_ref], check=True)
    subprocess.run(['git', 'clean', '-fd'], check=True)
    print('Synced to', origin_ref)
else:
    subprocess.run(['git', 'checkout', REPO_REF], check=True)
    subprocess.run(['git', 'reset', '--hard'], check=True)
    subprocess.run(['git', 'clean', '-fd'], check=True)
    print('No matching remote ref found for', origin_ref, '- keeping local checkout')

# Critical files required by current backend.main imports
required_files = [
    'backend/main.py',
    'backend/settings.py',
    'backend/studio_drafts.py',
    'backend/utils/groq.py',
]
missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise RuntimeError(f'Missing required backend files after checkout: {missing}')

subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['pip', 'install', '-r', 'backend/requirements.txt'], check=True)
subprocess.run(['pip', 'install', 'pyngrok', 'requests'], check=True)


In [ ]:
# ===== Entorno y storage =====
import os
from pathlib import Path
from getpass import getpass


def _read_colab_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value and str(value).strip():
            return str(value).strip()
    except Exception:
        pass
    return None


def _resolve_secret(name: str, explicit_value, required: bool = False):
    if explicit_value is not None and str(explicit_value).strip():
        return str(explicit_value).strip()

    from_colab_secret = _read_colab_secret(name)
    if from_colab_secret:
        return from_colab_secret

    if required:
        entered = getpass(f"{name}: ").strip()
        if not entered:
            raise RuntimeError(f"{name} es obligatorio para despliegue remoto seguro")
        return entered

    return None


if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = DRIVE_DATA_DIR
else:
    data_dir = LOCAL_DATA_DIR

Path(data_dir).mkdir(parents=True, exist_ok=True)

api_key = _resolve_secret('VOICEBOX_API_KEY', VOICEBOX_API_KEY, required=True)
groq_api_key = _resolve_secret('VOICEBOX_GROQ_API_KEY', VOICEBOX_GROQ_API_KEY, required=False)
ngrok_auth_token = _resolve_secret('NGROK_AUTH_TOKEN', NGROK_AUTH_TOKEN, required=False)

os.environ['VOICEBOX_COLAB_PROFILE'] = 'true'
os.environ['VOICEBOX_HOST'] = VOICEBOX_HOST
os.environ['VOICEBOX_PORT'] = VOICEBOX_PORT
os.environ['VOICEBOX_DEFAULT_MODEL_SIZE'] = VOICEBOX_DEFAULT_MODEL_SIZE
os.environ['VOICEBOX_STT_PROVIDER'] = VOICEBOX_STT_PROVIDER
os.environ['VOICEBOX_STT_REMOTE_NO_FALLBACK'] = VOICEBOX_STT_REMOTE_NO_FALLBACK
os.environ['VOICEBOX_GROQ_STT_MODEL'] = VOICEBOX_GROQ_STT_MODEL
os.environ['VOICEBOX_ALLOWED_ORIGINS'] = VOICEBOX_ALLOWED_ORIGINS
os.environ['VOICEBOX_DB_USE_NULL_POOL'] = VOICEBOX_DB_USE_NULL_POOL
os.environ['VOICEBOX_DATA_DIR'] = data_dir
os.environ['VOICEBOX_API_KEY'] = api_key

if groq_api_key:
    os.environ['VOICEBOX_GROQ_API_KEY'] = groq_api_key

print('VOICEBOX_DATA_DIR =', os.environ['VOICEBOX_DATA_DIR'])
print('VOICEBOX_COLAB_PROFILE =', os.environ['VOICEBOX_COLAB_PROFILE'])
print('VOICEBOX_PORT =', os.environ['VOICEBOX_PORT'])
print('VOICEBOX_DEFAULT_MODEL_SIZE =', os.environ['VOICEBOX_DEFAULT_MODEL_SIZE'])
print('VOICEBOX_STT_PROVIDER =', os.environ['VOICEBOX_STT_PROVIDER'])
print('VOICEBOX_STT_REMOTE_NO_FALLBACK =', os.environ['VOICEBOX_STT_REMOTE_NO_FALLBACK'])
print('VOICEBOX_GROQ_STT_MODEL =', os.environ['VOICEBOX_GROQ_STT_MODEL'])
print('VOICEBOX_DB_USE_NULL_POOL =', os.environ['VOICEBOX_DB_USE_NULL_POOL'])
print('VOICEBOX_API_KEY configurada =', bool(os.environ.get('VOICEBOX_API_KEY')))
print('VOICEBOX_GROQ_API_KEY configurada =', bool(os.environ.get('VOICEBOX_GROQ_API_KEY')))
if not os.environ.get('VOICEBOX_GROQ_API_KEY'):
    print('WARNING: VOICEBOX_GROQ_API_KEY no configurada. /transcribe respondera error hasta configurarla.')



In [ ]:
import os
# ===== Start backend =====
import subprocess
import time
import requests
from pathlib import Path

BASE_PORT = os.environ.get('VOICEBOX_PORT', '17493')
base = f"http://127.0.0.1:{BASE_PORT}"
headers = {'Authorization': f"Bearer {os.environ['VOICEBOX_API_KEY']}"}

required_files = [
    '/content/voicebox/backend/main.py',
    '/content/voicebox/backend/settings.py',
    '/content/voicebox/backend/studio_drafts.py',
    '/content/voicebox/backend/utils/groq.py',
]
missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise RuntimeError(f'Backend checkout incomplete. Missing files: {missing}. Re-run clone/install cell.')

log_path = '/tmp/voicebox_backend.log'
log_file = open(log_path, 'w')

server = subprocess.Popen([
    'python', '-m', 'uvicorn',
    'backend.main:app',
    '--host', os.environ.get('VOICEBOX_HOST', '0.0.0.0'),
    '--port', BASE_PORT,
], stdout=log_file, stderr=subprocess.STDOUT)

print('Server PID:', server.pid)
print('Backend logs:', log_path)

READY_TIMEOUT_SECONDS = 120
ready = False

for i in range(READY_TIMEOUT_SECONDS):
    if server.poll() is not None:
        log_file.close()
        print('Backend process exited early with code:', server.returncode)
        try:
            with open(log_path, 'r') as f:
                logs = f.read()[-8000:]
            print('\n=== Backend logs (tail) ===\n')
            print(logs)
        except Exception as e:
            print('Could not read backend logs:', e)
        raise RuntimeError('Backend crashed before becoming ready')

    try:
        r = requests.get(f"{base}/health", headers=headers, timeout=5)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass

    time.sleep(1)

if not ready:
    log_file.close()
    try:
        with open(log_path, 'r') as f:
            logs = f.read()[-8000:]
        print('\n=== Backend logs (tail) ===\n')
        print(logs)
    except Exception as e:
        print('Could not read backend logs:', e)
    raise RuntimeError('Backend did not become ready in time')

print('Backend ready:', base)
print('Quick health:', requests.get(f"{base}/health", headers=headers, timeout=10).json())


In [ ]:
import os
# ===== Public URL via ngrok =====
from pyngrok import ngrok

if ngrok_auth_token:
    ngrok.set_auth_token(ngrok_auth_token)

port = int(os.environ.get('VOICEBOX_PORT', '17493'))
ngrok.kill()  # cierra tuneles viejos para evitar colisiones
public_url = ngrok.connect(addr=f'127.0.0.1:{port}', proto='http', bind_tls=True).public_url

print('\n=== CONNECTION INFO ===')
print('Public URL:', public_url)
print('Server URL for app:', public_url)
print('API key: [HIDDEN]')
print('Header: Authorization: Bearer <API_KEY>')


In [ ]:
import os
# ===== Smoke tests =====
import io
import wave
import requests

base = f"http://127.0.0.1:{os.environ.get('VOICEBOX_PORT', '17493')}"
headers = {'Authorization': f"Bearer {os.environ['VOICEBOX_API_KEY']}"}

for path in ['/health', '/runtime', '/models/status']:
    resp = requests.get(base + path, headers=headers, timeout=60)
    print(path, '->', resp.status_code)
    print(resp.json())
    print('---')

# Optional STT smoke test (Groq remote)
if os.environ.get('VOICEBOX_GROQ_API_KEY'):
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(16000)
        wav.writeframes(bytes(2 * 16000))  # 1s silence
    buf.seek(0)

    files = {'file': ('smoke.wav', buf.getvalue(), 'audio/wav')}
    data = {'language': 'es'}
    resp = requests.post(f"{base}/transcribe", headers=headers, files=files, data=data, timeout=180)
    print('/transcribe (groq remote) ->', resp.status_code)
    try:
        print(resp.json())
    except Exception:
        print(resp.text)
else:
    print('/transcribe smoke test skipped: VOICEBOX_GROQ_API_KEY missing')




In [ ]:
import os
# ===== Optional: pre-cargar modelo 1.7B =====
# Ejecuta esta celda si quieres dejar el modelo listo antes de la primera generación.

import requests

base = f"http://127.0.0.1:{os.environ.get('VOICEBOX_PORT', '17493')}"
headers = {'Authorization': f"Bearer {os.environ['VOICEBOX_API_KEY']}"}

r = requests.post(f"{base}/models/load", params={'model_size': '1.7B'}, headers=headers, timeout=3600)
print('models/load ->', r.status_code)
print(r.json())
